# SAP SoD Conflict Detection - Exploratory Data Analysis

This notebook is used to test the **Segregation of Duties (SoD)** conflict detection logic in Pandas.

The `df_sod` risk matrix mirrors the **SAP GRC Access Control (Access Rule Maintenance)** standard and ISACA/SOX 404 internal control principles (detecting *Four-Eyes Principle* violations in the Procure-to-Pay process).

## Libraries Import

In [1]:
import os
import random
import pandas as pd

## SoD Risk Matrix & Internal Control Framework

The transaction codes (T-Codes) and conflict rules defined below strictly follow **SAP GRC Access Control (ARA)** specifications and **ISACA ERP Audit Guidelines**:

* **Transaction Codes (T-Codes):** Standard actions across SAP FI/MM modules, including `FK01` (Vendor Creation), `MIRO` (Invoice Processing), `FK05` (Vendor Block/Unblock), and `PAY01`/`F110` (Payment Execution).
* **SoD Risk Rules (`df_sod`):** Severity levels (`CRITICAL`, `HIGH`) mapped according to the **Four-Eyes Principle** to prevent overlapping Authorization, Execution, and Custody duties in the Procure-to-Pay (P2P) process.

In [ ]:

sod_rules = [
    {
        #authorization + execution 
        "risk_id": "SOD_01", # polaczenie autoryzacji i wykonania;  ktos tworzy fikcyjnego dostawce i mu wysyla kase
        "tcode_a": "FK01",  # create vendor - tworzenie dostawcy 
        "tcode_b": "PAY01",  # approve payment 
        "risk_level": "CRITICAL",
        "description": "Creating vendor and approving payment.",
    },
    {
        #recording + execution 
        "risk_id": "SOD_02", # rozdzielenie funckji rejestracji recording od execution czyli wykonania platnosci; ktos wprowadza zwobowazania i wyplaca gotowke
        "tcode_a": "FB01",  # post entry - wpis księgowy 
        "tcode_b": "PAY01",  # approve payment 
        "risk_level": "HIGH",
        "description": "Posting entries and executing payment run.",
    },
    {
        # authorization + recording
        "risk_id": "SOD_03", # odpowiada za procesy wewnatrz lancucha dostaw i zakupow - sap materials menagment; rozdzielenie procesu zamawiania od ksiegowania faktyr zakupwych; ktos sklada zamowienie i zatweirdza fakture nawet jak produkt nei dotarl  
        "tcode_a": "ME21N",  # create po - stworzenie purchase order
        "tcode_b": "MIRO",  # enter invoice; rejestracja/wpis ksiegowy 
        "risk_level": "MEDIUM",
        "description": "Creating PO and entering invoice.",
    },
]

df_sod = pd.DataFrame(sod_rules)

print("SoD Risk Matrix")
display(df_sod)

MACIERZ RYZYK SoD


,risk_id,tcode_a,tcode_b,risk_level,description
0,SOD_01,FK01,PAY01,CRITICAL,Creating vendor and approving payment.
1,SOD_02,FB01,PAY01,HIGH,Posting entries and executing payment run.
2,SOD_03,ME21N,MIRO,MEDIUM,Creating PO and entering invoice.


# User Roles Dataset

In [4]:
user_roles_data = [
    {"user_id": "U001", "tcode": "FK01", "role_name": "Vendor_Admin"}, # zakladanie i edycja dostawcow
    {"user_id": "U001", "tcode": "PAY01", "role_name": "Payment_Clerk"}, # zatwierdzanie i realizacja przelewow
    {"user_id": "U002", "tcode": "FB01", "role_name": "GL_Accountant"}, # ksiegowy ksiegi glownej 3recznie wprowadza wpisy ksiegowe i koretky na kontach 
    {"user_id": "U002", "tcode": "PAY01", "role_name": "Payment_Clerk"}, 
    {"user_id": "U003", "tcode": "ME21N", "role_name": "Purchaser"}, # towry oficjalne zamowienia sklepu purchase orde
    {"user_id": "U003", "tcode": "MIRO", "role_name": "AP_Clerk"},   #ksiegowy  ds zobowiazan accounts payable, przyjmuje i rejestruje faktury zakupowe    
    {"user_id": "U004", "tcode": "FK01", "role_name": "Vendor_Admin"},  
]

df_user_roles = pd.DataFrame(user_roles_data)

print("User Roles Dataset")
display(df_user_roles)

User Roles Dataset


,user_id,tcode,role_name
0,U001,FK01,Vendor_Admin
1,U001,PAY01,Payment_Clerk
2,U002,FB01,GL_Accountant
3,U002,PAY01,Payment_Clerk
4,U003,ME21N,Purchaser
5,U003,MIRO,AP_Clerk
6,U004,FK01,Vendor_Admin


# Self Merge


In [8]:
df_pairs = pd.merge(
    df_user_roles, 
    df_user_roles, 
    on="user_id", 
    suffixes=("_a", "_b")
) # merging roles table on user id column 


df_pairs = df_pairs[df_pairs["tcode_a"] != df_pairs["tcode_b"]] # delating pairs where transaction is merget with itself

print("Transaction pairs for each user")
display(df_pairs)

Transaction pairs for each user


,user_id,tcode_a,role_name_a,tcode_b,role_name_b
1,U001,FK01,Vendor_Admin,PAY01,Payment_Clerk
2,U001,PAY01,Payment_Clerk,FK01,Vendor_Admin
5,U002,FB01,GL_Accountant,PAY01,Payment_Clerk
6,U002,PAY01,Payment_Clerk,FB01,GL_Accountant
9,U003,ME21N,Purchaser,MIRO,AP_Clerk
10,U003,MIRO,AP_Clerk,ME21N,Purchaser


# Detecting Violation SoD by Matching Pairs to Risk Matrix 

In [9]:
df_conflicts = pd.merge(
    df_pairs,
    df_sod,
    on=["tcode_a", "tcode_b"], # wskazujemy ktore kolumny musza do siebie idealnie pasowac w obu tabelach zeby doszlo do polaczenia wierszy 
    how="inner" #zostawia tylko te wiersze ktore znalazly dopasowanie w obu tabelach 
)

df_conflicts = df_conflicts[
    ["user_id", "risk_id", "risk_level", "tcode_a", "tcode_b", "role_name_a", "role_name_b", "description"]
] # chosing columns - which and in which order to the audit report 

print("Detected Violation of SoD")
display(df_conflicts)

Detected Violation of SoD


,user_id,risk_id,risk_level,tcode_a,tcode_b,role_name_a,role_name_b,description
0,U001,SOD_01,CRITICAL,FK01,PAY01,Vendor_Admin,Payment_Clerk,Creating vendor and approving payment.
1,U002,SOD_02,HIGH,FB01,PAY01,GL_Accountant,Payment_Clerk,Posting entries and executing payment run.
2,U003,SOD_03,MEDIUM,ME21N,MIRO,Purchaser,AP_Clerk,Creating PO and entering invoice.


# Export SoD analysis results to Excel report with two sheets -one qith user authorisations and one with report 

In [ ]:
output_file = "sod_analysis_report_test.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_user_roles.to_excel(
        writer, sheet_name="User Authorizations", index=False
    )
    df_conflicts.to_excel(
        writer, sheet_name="SoD Conflicts Report", index=False
    )

print(f"Done - found {len(df_conflicts)} conflicts. Saved to {output_file}")

Done. Found 3 conflicts. Saved to sod_analysis_report.xlsx
